# Imaging low-trap: best (f, amp) by heating + detection fidelity

Interactive analysis of `ImagingLowTrapScan` runs. Reuses the helpers from `analyze_imaging_lowtrap.py`:

- **Metric 1 (heating)** - survival@0 (survives two 4 ms images) and survival@release_probe (temperature sensitivity).
- **Metric 2 (separation)** - double-Gaussian threshold between shot1 (atom) and bg (no atom) -> fidelity = 0.5*(P_detect + P_reject).

Best (f, amp) = max fidelity with survival@0 >= floor. **Run with the working-BLAS `python` kernel** (curve_fit / matplotlib), NOT artiq7_claude (broken BLAS).

In [ ]:
import sys, os, glob
from pathlib import Path
_ANALYSIS_DIR = r"C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\analysis"
if _ANALYSIS_DIR not in sys.path:
    sys.path.insert(0, _ANALYSIS_DIR)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%load_ext autoreload
%autoreload 2
from analyze_imaging_lowtrap import (
    RESULTS_DIR, SURVIVAL_FLOOR, find_run, load_run, wilson, _iqr,
    build_cells, cell_fidelity, cell_survival, build_grids,
)
print("Results dir   :", RESULTS_DIR)
print("Survival floor:", SURVIVAL_FLOOR)

## 1. List recent runs

In [ ]:
files = sorted(glob.glob(str(RESULTS_DIR / "**" / "*-ImagingLowTrapScan.h5"), recursive=True),
               key=lambda f: int(os.path.basename(f).split("-")[0]))
if not files:
    print("No *-ImagingLowTrapScan.h5 found yet.")
for f in files[-12:]:
    rid = int(os.path.basename(f).split("-")[0])
    d, s, _ = load_run(f)
    n = len(np.atleast_1d(d.get("img_f", [])))
    nl = int(np.sum(np.atleast_1d(d.get("img_loaded", [0])).astype(int)))
    print(f"RID {rid:>5} | {n:>6} pts | {nl:>5} loaded | precool={s.get('use_pgc_precool','?')}")

## 2. Select a run

Set `RID` to an int, or `None` for the latest.

In [ ]:
RID = None   # <-- int or None (latest)

path = find_run(RID)
data, snap, rid = load_run(path)
cells = build_cells(data)
(fs, amps, M_surv0, M_survP, M_fid, M_gap, rel_probe, best_fa, best_fid) = build_grids(data, snap)
fs_mhz = [f/1e6 for f in fs]

print("File :", os.path.basename(path))
print(f"RID {rid} | {len(fs)} freq x {len(amps)} amp | release_probe={rel_probe:g} us")
print(f"baseline imaging: f_cooling_imaging={float(snap.get('f_cooling_imaging',0))/1e6:.4f} MHz, "
      f"amp_cooling_imaging={snap.get('amp_cooling_imaging','?')}")

## 3. Metric 1 - heating / survival

In [ ]:
fig, (ax0, axP) = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
for ax, M, ttl in [(ax0, M_surv0, f"survival @ 0 us  (RID {rid})"),
                   (axP, M_survP, f"survival @ {rel_probe:g} us (temperature)")]:
    im = ax.pcolormesh(fs_mhz, amps, M, shading="nearest", cmap="viridis", vmin=0.0, vmax=1.0)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if best_fa is not None:
        ax.plot(best_fa[0]/1e6, best_fa[1], "*", color="red", ms=15, markeredgecolor="white")
    ax.set_xlabel("imaging-DP frequency (MHz)"); ax.set_ylabel("imaging-DP amplitude"); ax.set_title(ttl)
plt.show()

## 4. Metric 2 - detection fidelity + histogram gap

In [ ]:
fig, (axF, axG) = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
imF = axF.pcolormesh(fs_mhz, amps, M_fid, shading="nearest", cmap="viridis", vmin=0.5, vmax=1.0)
plt.colorbar(imF, ax=axF, fraction=0.046, pad=0.04)
imG = axG.pcolormesh(fs_mhz, amps, M_gap, shading="nearest", cmap="magma")
plt.colorbar(imG, ax=axG, fraction=0.046, pad=0.04)
for ax, ttl in [(axF, "detection fidelity 0.5*(P_detect+P_reject)"), (axG, "histogram gap median(shot1)-median(bg)")]:
    if best_fa is not None:
        ax.plot(best_fa[0]/1e6, best_fa[1], "*", color="red", ms=15, markeredgecolor="white")
    ax.set_xlabel("imaging-DP frequency (MHz)"); ax.set_ylabel("imaging-DP amplitude"); ax.set_title(ttl)
plt.show()
if best_fa is not None:
    print(f"BEST (f, amp) = {best_fa[0]/1e6:.4f} MHz, {best_fa[1]:.3f}  fidelity={best_fid:.3f}")
else:
    print(f"No (f,amp) with survival@0 >= {SURVIVAL_FLOOR:.0%}: imaging ejects the atom everywhere "
          "at this gate/trap (shorten the gate or use a less shallow trap).")

## 5. Best cell - shot1 (atom) vs bg (no atom) histogram

In [ ]:
if best_fa is not None:
    c = cells[best_fa]
    fid, thr, gap, sep = cell_fidelity(c["shot1"], c["bg"])
    allc = np.concatenate([c["shot1"], c["bg"]])
    bins = np.linspace(float(allc.min()), float(allc.max()), 31)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(c["bg"], bins=bins, color="C0", alpha=0.6, label="bg (no atom)")
    ax.hist(c["shot1"], bins=bins, color="C3", alpha=0.6, label="shot1 (atom)")
    ax.axvline(thr, color="k", ls="--", lw=1.5, label=f"threshold {thr:.0f}")
    ax.set_xlabel("SPCM1+SPCM2 counts"); ax.set_ylabel("occurrences")
    ax.set_title(f"Best f={best_fa[0]/1e6:.3f} MHz amp={best_fa[1]:.3f}  fid={fid:.3f}  sep={sep:.2f}")
    ax.legend(); plt.show()
else:
    print("No best cell (survival floor not met).")

## 6. Full picture — every (f, amp) histogram in one grid

shot1 (atom, red) vs bg (no atom, blue), dashed = the per-cell double-Gaussian threshold; `F` = fidelity, `s` = survival@0. Rows = imaging amplitude (high at top, matching the heatmaps), cols = imaging frequency.

In [ ]:
# Full picture: every (f, amp) histogram in one grid (shot1 atom vs bg no-atom + per-cell threshold)
_allc = np.concatenate([np.concatenate([c["shot1"], c["bg"]]) for c in cells.values()])
_bins = np.linspace(float(_allc.min()), float(_allc.max()), 31)
_nf, _na = len(fs), len(amps)
fig, axes = plt.subplots(_na, _nf, sharex=True, sharey=True, squeeze=False,
                         figsize=(max(1.5*_nf, 6), max(1.4*_na, 4)))
for _i in range(_na):
    _a = amps[_na - 1 - _i]                      # row 0 = highest amp (match the heatmaps)
    for _j in range(_nf):
        _f = fs[_j]
        ax = axes[_i][_j]
        c = cells.get((_f, _a))
        if c is None:
            ax.axis("off"); continue
        _fid, _thr, _gap, _sep = cell_fidelity(c["shot1"], c["bg"])
        _s0 = cell_survival(c, 0.0, _thr)
        ax.hist(c["bg"], bins=_bins, color="C0", alpha=0.6)
        ax.hist(c["shot1"], bins=_bins, color="C3", alpha=0.6)
        if np.isfinite(_thr):
            ax.axvline(_thr, color="k", ls="--", lw=0.7)
        _s0t = f"{_s0[0]:.2f}" if _s0 is not None else "-"
        ax.text(0.96, 0.94, f"F{_fid:.2f}\ns{_s0t}", transform=ax.transAxes, ha="right", va="top",
                fontsize=5, bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6))
        ax.tick_params(labelsize=4, length=2)
        if _j == 0:
            ax.set_ylabel(f"a={_a:.3f}", fontsize=6)
        if _i == 0:
            ax.set_title(f"{_f/1e6:.2f}", fontsize=6)
fig.suptitle(f"Per-(f,amp) histograms: shot1(atom,red) vs bg(no-atom,blue), dashed=threshold -- RID {rid}\n"
             "cols=freq MHz, rows=amp; F=fidelity, s=survival@0", fontsize=10)
plt.show()

## 7. (optional) one-shot PNGs via the script

`analyze(path)` writes the heatmaps + best-cell histogram + the **full histogram grid** PNGs next to the `.h5`.

In [ ]:
# from analyze_imaging_lowtrap import analyze
# analyze(path)